# MobileNetV2: Inverted Residual Bottlenecks and Depthwise Separable Convolutions
### A Ground-Up PyTorch Implementation for Masters-Level Practitioners

---

> *"The most important thing is not to stop questioning. Curiosity has its own reason for existing."*
> — Albert Einstein

---

## Why This Notebook Exists

In 2017, Google researchers published [MobileNetV1](https://arxiv.org/abs/1704.04861), introducing depthwise separable convolutions as a drop-in replacement for standard convolutions in mobile and edge settings. A year later, [MobileNetV2](https://arxiv.org/abs/1801.04381) went further — it introduced the **Inverted Residual Bottleneck**, a building block that is now one of the most widely deployed architectural primitives in production computer vision.

Yet despite its ubiquity, the design rationale behind MobileNetV2 is often taught at the wrong level of abstraction. Papers reference formulas, `torchvision` ships a black-box module, and tutorials tend to skip over the precise tensor mechanics that make the architecture tick.

This notebook refuses to do that.

We are going to build everything from **raw PyTorch primitives** — `nn.Conv2d`, `nn.BatchNorm2d`, `nn.ReLU6` — and we will trace every tensor transformation with explicit `[B, C, H, W]` annotations. By the end, you will understand not just *what* MobileNetV2 does, but *why* each design decision was made, and what trade-off it encodes.

---

## Roadmap

| Stage | Topic | Key Concept |
|---|---|---|
| **1** | Standard Convolution | Baseline cost & parameter count |
| **2** | Depthwise Separable Convolution | Factorized spatial + channel mixing |
| **3** | Linear Bottlenecks & ReLU6 | Information preservation under quantization |
| **4** | Inverted Residual Block | Expand → Depthwise → Project |
| **5** | Full MobileNetV2 Backbone | Stacking blocks with stride control |
| **6** | FLOP & parameter analysis | Efficiency benchmarking vs. ResNet |

This notebook covers **Stages 1–4** in full.

---

## Prerequisites

You should be comfortable with:
- PyTorch `nn.Module` anatomy (forward pass, parameter registration)
- Convolution arithmetic: output size given kernel, stride, padding
- Batch normalization mechanics (running stats, training vs. eval mode)

If convolution arithmetic feels shaky, revisit [CS231n Lecture 5](http://cs231n.stanford.edu/slides/2022/lecture_5.pdf) before proceeding.

---

## Environment Setup

We pin exact versions to ensure reproducibility. The notebook was developed and tested against:

```
torch==2.3.0
torchsummary==1.5.1
matplotlib==3.8.4
numpy==1.26.4
```

We deliberately **do not** import `torchvision.models`. Every layer will be authored from scratch using `torch.nn` primitives only.

In [ ]:
from __future__ import annotations

import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

---

## Part 1 — The Standard Convolution: Our Baseline

Before we can appreciate what MobileNetV2 is doing, we need a precise accounting of what a standard convolution *costs*. Let's be rigorous.

### 1.1 Anatomy of a Conv-BN-ReLU Block

The standard building block in most CNNs is not a bare convolution — it is the **Conv → BatchNorm → Activation** triplet. Each component plays a specific role:

- **`nn.Conv2d`** — Applies learned spatial filters. For input `[B, C_in, H, W]` and `C_out` filters of shape `[K, K]`, the output is `[B, C_out, H', W']`.
- **`nn.BatchNorm2d`** — Normalizes each channel's activations across the batch dimension. This stabilizes training and allows higher learning rates. It introduces two *learnable* affine parameters per channel: scale `γ` and shift `β`.
- **Activation** — Introduces non-linearity. Standard CNNs use ReLU; MobileNetV2 uses **ReLU6** (`min(max(x, 0), 6)`) for numerical stability under fixed-point quantization.

### 1.2 Parameter Count

For a standard convolution with:
- Input channels: $C_{in}$
- Output channels: $C_{out}$  
- Kernel size: $K \times K$
- Bias: disabled (standard practice with BatchNorm)

$$\text{Parameters}_{\text{conv}} = C_{out} \times C_{in} \times K \times K$$

BatchNorm adds $2 \times C_{out}$ learnable parameters ($\gamma$ and $\beta$), plus $2 \times C_{out}$ non-learnable running statistics (mean and variance). For large channel counts this is negligible relative to the convolution kernel.

### 1.3 FLOP Count

For each output spatial location $(h', w')$, computing one output channel requires $C_{in} \times K \times K$ multiply-accumulate (MAC) operations. With $C_{out}$ output channels and $H' \times W'$ output spatial positions:

$$\text{MACs}_{\text{std}} = H' \times W' \times C_{in} \times C_{out} \times K^2$$

This is the cost we will dramatically reduce through factorization.

In [ ]:
class StandardConv(nn.Module):
    """
    Standard Conv2d → BatchNorm2d → ReLU6 block.

    Tensor flow
    -----------
    Input  : [B, C_in,  H,  W]
    Conv2d : [B, C_out, H', W']   where H' = floor((H + 2P - K) / S) + 1
    BN     : [B, C_out, H', W']
    ReLU6  : [B, C_out, H', W']

    Parameters
    ----------
    in_channels  : Number of channels in the input feature map.
    out_channels : Number of convolutional filters (output channels).
    kernel_size  : Spatial extent of each filter. Default 3 (3×3).
    stride       : Step size of the sliding window. stride=2 halves H and W.
    padding      : Zero-padding added to both spatial dims.
    groups       : Controls filter grouping (1 = standard conv).
    bias         : Disabled — BatchNorm absorbs the bias term.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        groups: int = 1,
        bias: bool = False,
    ) -> None:
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size,
            stride=stride, padding=padding, groups=groups, bias=bias,
        )
        self.bn  = nn.BatchNorm2d(out_channels, eps=1e-5, momentum=0.1, affine=True)
        self.act = nn.ReLU6(inplace=True)
        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.kaiming_uniform_(self.conv.weight, mode="fan_out", nonlinearity="relu")
        nn.init.ones_(self.bn.weight)
        nn.init.zeros_(self.bn.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)  # [B, C_in, H, W]  → [B, C_out, H', W']
        x = self.bn(x)    # [B, C_out, H', W'] (normalize)
        x = self.act(x)   # [B, C_out, H', W'] (clip to [0, 6])
        return x

    @staticmethod
    def count_params(in_channels: int, out_channels: int, kernel_size: int = 3) -> int:
        return out_channels * in_channels * kernel_size * kernel_size

    @staticmethod
    def count_macs(in_channels, out_channels, h_out, w_out, kernel_size=3) -> int:
        return h_out * w_out * in_channels * out_channels * kernel_size ** 2

In [ ]:
B, C_in, H, W = 4, 32, 112, 112
C_out = 64
K = 3

std_conv = StandardConv(C_in, C_out, K, stride=1, padding=1).to(DEVICE)
x_in     = torch.randn(B, C_in, H, W, device=DEVICE)
x_out    = std_conv(x_in)

assert x_out.shape == (B, C_out, H, W)
assert x_out.min() >= 0.0 and x_out.max() <= 6.0

H_out, W_out = x_out.shape[2], x_out.shape[3]
print(f"StandardConv  {list(x_in.shape)} → {list(x_out.shape)}")
print(f"  Params : {StandardConv.count_params(C_in, C_out, K):,}")
print(f"  MACs   : {StandardConv.count_macs(C_in, C_out, H_out, W_out, K)/1e6:.2f}M")
print(f"  Range  : [{x_out.min():.3f}, {x_out.max():.3f}]  ✓")

---

## Part 2 — Depthwise Separable Convolution: The Key Factorization

### 2.1 The Core Insight

A standard convolution does two fundamentally different things in a single operation:

1. **Spatial filtering** — convolves each output channel with a $K \times K$ receptive field.
2. **Channel mixing** — projects from $C_{in}$ to $C_{out}$ channels.

These are orthogonal operations and can be factorized:

| Step | Operation | Role | Cost |
|---|---|---|---|
| **Depthwise (DW)** | `groups=C_in` Conv2d | Spatial filtering per channel | $H'W' C_{in} K^2$ MACs |
| **Pointwise (PW)** | `kernel_size=1` Conv2d | Channel mixing | $H'W' C_{in} C_{out}$ MACs |

Reduction ratio: $\dfrac{1}{C_{out}} + \dfrac{1}{K^2}$ — roughly $8\times$ for $K=3$, $C_{out}=64$.

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    """
    Depthwise Separable Convolution: Depthwise → Pointwise.

    Tensor flow
    -----------
    Input       : [B, C_in,  H,  W ]
    ── Depthwise ────────────────────────────────────────────────────────────
    DW Conv2d   : [B, C_in,  H', W']  groups=C_in; one 3×3 filter per channel
    DW BN       : [B, C_in,  H', W']
    DW ReLU6    : [B, C_in,  H', W']
    ── Pointwise ────────────────────────────────────────────────────────────
    PW Conv2d   : [B, C_out, H', W']  kernel=1×1; full cross-channel projection
    PW BN       : [B, C_out, H', W']
    PW ReLU6    : [B, C_out, H', W']
    """

    def __init__(self, in_channels: int, out_channels: int,
                 stride: int = 1, dw_padding: int = 1) -> None:
        super().__init__()

        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, stride=stride,
                      padding=dw_padding, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels, eps=1e-5, momentum=0.1, affine=True),
            nn.ReLU6(inplace=True),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride=1, padding=0,
                      groups=1, bias=False),
            nn.BatchNorm2d(out_channels, eps=1e-5, momentum=0.1, affine=True),
            nn.ReLU6(inplace=True),
        )
        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)   # [B, C_in, H, W]  → [B, C_in,  H', W']
        x = self.pointwise(x)  # [B, C_in, H', W'] → [B, C_out, H', W']
        return x

    @staticmethod
    def count_params(in_channels, out_channels, kernel_size=3) -> dict:
        dw = in_channels * kernel_size * kernel_size
        pw = out_channels * in_channels
        return {"depthwise": dw, "pointwise": pw, "total": dw + pw}

    @staticmethod
    def count_macs(in_channels, out_channels, h_out, w_out, kernel_size=3) -> dict:
        dw = h_out * w_out * in_channels * kernel_size ** 2
        pw = h_out * w_out * in_channels * out_channels
        return {"depthwise": dw, "pointwise": pw, "total": dw + pw}

In [ ]:
dsc       = DepthwiseSeparableConv(C_in, C_out, stride=1).to(DEVICE)
x_out_dsc = dsc(x_in)
assert x_out_dsc.shape == (B, C_out, H, W)

std_p = StandardConv.count_params(C_in, C_out, K)
dsc_p = DepthwiseSeparableConv.count_params(C_in, C_out, K)
std_m = StandardConv.count_macs(C_in, C_out, H, W, K)
dsc_m = DepthwiseSeparableConv.count_macs(C_in, C_out, H, W, K)

print(f"{'Metric':30s} {'StandardConv':>14s}  {'DSConv':>14s}  {'Ratio':>8s}")
print("-" * 72)
print(f"{'Parameters':30s} {std_p:>14,}  {dsc_p['total']:>14,}  {dsc_p['total']/std_p:>7.3f}x")
print(f"{'MACs (M)':30s} {std_m/1e6:>13.2f}M  {dsc_m['total']/1e6:>13.2f}M  {dsc_m['total']/std_m:>7.3f}x")
print(f"\nTheoretical reduction: 1/{C_out} + 1/{K}² = {1/C_out + 1/K**2:.4f}  "
      f"({1/(1/C_out + 1/K**2):.2f}× cheaper)")

---

## Part 3 — The Linear Bottleneck: Why We Drop the Activation

Before we can implement the inverted residual block, we need to confront one of the most subtle design decisions in MobileNetV2 — one that is frequently glossed over: the **deliberate removal of the non-linear activation at the projection stage**. This is not a minor implementation detail. It is a principled architectural choice rooted in the geometry of high-dimensional representations, and understanding it is essential for understanding why the block works.

### 3.1 The Manifold Hypothesis in Deep Networks

A well-established empirical observation in deep learning — formalized as the **manifold hypothesis** — states that the set of activations produced by a trained network does not fill its ambient space. Instead, activations cluster on a low-dimensional **manifold** embedded within that high-dimensional space.

More precisely: suppose the layer just before a bottleneck produces activations in $\mathbb{R}^{C}$ where $C = 512$. The manifold hypothesis asserts that the "interesting" directions — the ones that encode discriminative information — span only a small subspace of dimension $d \ll C$. The rest is either noise or redundancy introduced by over-parameterisation.

This is the justification for bottleneck layers at all: if intrinsic dimensionality is $d$, we can project to $d$ dimensions without destroying information. The standard convolution is doing far more work than necessary.

### 3.2 The Catastrophic Effect of ReLU on Low-Dimensional Embeddings

Here is the problem. The manifold hypothesis justifies *compression* — projecting to a lower-dimensional space — but it does **not** justify applying a non-linearity *after* that compression.

Consider what ReLU does geometrically. For a vector $\mathbf{z} \in \mathbb{R}^d$:
$$\text{ReLU}(\mathbf{z}) = \max(\mathbf{z},\, \mathbf{0})$$

This partitions $\mathbb{R}^d$ into $2^d$ orthants and **collapses every negative-component region onto the boundary of the positive orthant**. Two distinct points $\mathbf{z}_1 \neq \mathbf{z}_2$ that differ only in sign in some coordinates become identical after ReLU — and there is no way to recover which input produced which output.

**When $d$ is large** — say, $d=256$ — this is usually benign. The manifold is spread across many dimensions; the probability that two nearby manifold points collapse to the same output is low, because there is enough redundancy in the remaining dimensions to distinguish them.

**When $d$ is small** — say, $d=6$ — there is no such redundancy. A substantial fraction of the manifold may map to the exact same zero vector under ReLU, because most of its components are negative and zeroed out. The MobileNetV2 paper formalises this as:

> *If the manifold of interest is a low-dimensional subspace, ReLU is guaranteed to lose information unless the subspace lies entirely within a positive orthant of the input space.*

This is not a theoretical curiosity — it directly affects training dynamics. When the projection bottleneck has only 16 or 24 channels and a ReLU follows, a large fraction of feature directions are permanently zeroed. Gradients cannot flow back through zeroed units, starving the preceding layers of learning signal. The network appears to train, but the bottleneck is a lossy compressor rather than a faithful one.

### 3.3 The Fix: A Linear Projection Stage

The solution is to make the final projection a **linear map**: Conv2d followed by BatchNorm2d, but with **no activation function**.

BatchNorm is an affine transform — it scales and shifts each channel by learnable parameters $\gamma$ and $\beta$. An affine transform is bijective on its domain (it can be inverted); it does not zero out any direction of the manifold. So BatchNorm without activation is safe at the bottleneck.

The necessary non-linearities are provided by:
1. The **expansion stage** (ReLU6 after the 1×1 expansion conv), which operates in the wide $tN$-dimensional space.
2. The **depthwise stage** (ReLU6 after the 3×3 DW conv), which also operates in the wide space.

Both of these stages have high enough dimensionality that ReLU's collapsing behaviour is benign. The manifold is only compressed back to the narrow bottleneck *after* all non-linearities are complete — and at that point, linearity is exactly what we want.

### 3.4 Empirical Demonstration: Information Collapse Under ReLU

Let us make this concrete. We will construct a 1D smooth manifold (a parametric curve) embedded in a 512-dimensional space, project it to bottlenecks of varying dimensionality, apply/omit ReLU, then measure reconstruction error by projecting back with the pseudoinverse. If ReLU collapses distinct manifold points together, reconstruction error will be high.

In [ ]:
import numpy as np

torch.manual_seed(SEED)

def relu_collapse_experiment(
    ambient_dim: int = 512,
    bottleneck_dims: list = None,
    n_manifold_points: int = 2000,
) -> None:
    """
    Measures information loss when a 1D manifold is projected into a
    low-dimensional bottleneck with and without ReLU.

    Protocol
    --------
    1. Sample points on a smooth 1D curve in R^ambient_dim.
    2. Project down to R^d via a random Gaussian matrix W  (shape [d, ambient_dim]).
    3. Apply ReLU (or not) to the projected points.
    4. Reconstruct by applying the pseudoinverse W^+ of W.
    5. Report mean squared reconstruction error, normalised by signal power.
    """
    if bottleneck_dims is None:
        bottleneck_dims = [2, 4, 8, 16, 32, 64, 128]

    # ── Manifold: a smooth Lissajous curve lifted into R^ambient_dim ──────────
    t = torch.linspace(0, 2 * math.pi, n_manifold_points)  # [N]
    # 2D curve: (sin(t), cos(2t))
    curve_2d = torch.stack([torch.sin(t), torch.cos(2 * t)], dim=1)  # [N, 2]
    # Embed into ambient_dim via a fixed random linear map
    embed_matrix = torch.randn(2, ambient_dim) / math.sqrt(2)  # [2, D]
    X = curve_2d @ embed_matrix  # [N, D]  — manifold in R^D
    signal_power = X.pow(2).mean().item()

    print(f"Manifold: 1D Lissajous curve in R^{ambient_dim}")
    print(f"Signal power: {signal_power:.4f}")
    print()
    print(f"{'Bottleneck d':>13s}  {'Linear err':>12s}  {'ReLU err':>12s}  {'Collapse factor':>15s}")
    print("-" * 60)

    for d in bottleneck_dims:
        # Random projection matrix: [d, D]
        W = torch.randn(d, ambient_dim) / math.sqrt(ambient_dim)

        # Project: [N, d]
        Z = X @ W.T

        # Pseudoinverse for reconstruction: [D, d]
        W_pinv = torch.linalg.pinv(W)  # [D, d]

        # ── Linear bottleneck (no activation) ────────────────────────────────
        X_recon_linear = Z @ W_pinv.T                          # [N, D]
        err_linear = (X - X_recon_linear).pow(2).mean().item() / signal_power

        # ── Non-linear bottleneck (ReLU applied in low-dim space) ────────────
        Z_relu         = torch.relu(Z)                         # [N, d]  — collapse!
        X_recon_relu   = Z_relu @ W_pinv.T                     # [N, D]
        err_relu       = (X - X_recon_relu).pow(2).mean().item() / signal_power

        collapse = err_relu / (err_linear + 1e-12)
        print(f"  d = {d:5d}       {err_linear:>10.4f}    {err_relu:>10.4f}    {collapse:>12.2f}×")


relu_collapse_experiment()

**Reading the results:** The "collapse factor" is the ratio of ReLU reconstruction error to linear reconstruction error. At high bottleneck dimensions ($d \geq 64$), the two are comparable — ReLU's zeroing is not catastrophic when the representation is high-dimensional. But as $d$ shrinks toward 2–8, the ReLU error explodes relative to the linear baseline. This is information collapse in action: the non-linearity is permanently destroying manifold structure at the bottleneck.

The takeaway is architectural policy: **any layer that projects activations into a space with fewer dimensions than the intrinsic dimensionality of the manifold should not be followed by a ReLU**. In MobileNetV2, the projection layer compresses from $tN$ (e.g. 96) down to $N'$ (e.g. 16) — a compression of $6\times$. The linear bottleneck is the direct, principled response to this risk.

---

## Part 4 — From Standard Residuals to Inverted Residual Bottlenecks

### 4.1 The Standard ResNet Bottleneck: What We're Departing From

He et al.'s residual connection introduced what may be the single most important structural primitive in modern deep learning:
$$\mathbf{y} = \mathcal{F}(\mathbf{x},\, \{W_i\}) + \mathbf{x}$$

The additive shortcut serves two purposes simultaneously. First, it provides a **gradient highway** — during backpropagation, gradients can flow directly from loss to early layers without passing through the full chain of non-linearities. Second, it encourages the block to learn a **residual perturbation** $\mathcal{F}(\mathbf{x}) = \mathbf{y} - \mathbf{x}$ rather than a full mapping, which is easier when the optimal transform is close to identity.

ResNet-50's bottleneck for a layer with $C$ channels looks like this:

```
  Input : [B,   C,   H, W]  (C = 256 for the first bottleneck)
    │
    ├── 1×1 Conv → BN → ReLU : [B, C/4, H, W]   ← compress to narrow
    │   3×3 Conv → BN → ReLU : [B, C/4, H, W]   ← spatial mix (in narrow space)
    │   1×1 Conv → BN → ReLU : [B,   C, H, W]   ← expand back to wide
    │
    └─(+) shortcut            : [B,   C, H, W]   ← residual on WIDE ends

  Output: [B,   C,   H, W]
```

The key architectural fact: the expensive 3×3 convolution operates in the **narrow** $C/4$-channel space. The shortcut connects the **wide** $C$-channel ends. This is a purely computation-driven choice — making the 3×3 conv cheap by reducing its channel count.

### 4.2 The Structural Inversion: MobileNetV2's Key Insight

MobileNetV2 inverts this structure. The reasoning has two parts.

**Part 1: The depthwise convolution is already cheap regardless of channel count.** Recall that the DW conv has cost $H'W' \cdot C_{in} \cdot K^2$ — it scales linearly with $C_{in}$, not quadratically. There is therefore no benefit to performing the DW conv in a narrow channel space; it is cheap in any space. Keeping the DW conv in a *wide* space gives it access to a richer representation without proportionally more cost.

**Part 2: The channel-mixing operations (1×1 convolutions) are cheap precisely because they have no spatial extent.** If we want to reduce cost, the lever is the *channel dimension* of the DW conv, not the 1×1 convs. Expanding before the DW gives richer spatial filtering; projecting after it compresses back to memory-efficient bottleneck representations.

The resulting structure — Expand → Depthwise → Project — is the **inverted residual**:

```
  Input : [B,  N,   H, W]   (N = narrow bottleneck, e.g. 16 channels)
    │
    ├── 1×1 Conv → BN → ReLU6 : [B, tN,  H, W]  ← EXPAND to wide (t=6 → 96ch)
    │   3×3 DW  → BN → ReLU6 : [B, tN, H', W']  ← spatial mix in WIDE space
    │   1×1 Conv → BN         : [B,  N', H', W'] ← PROJECT to narrow (NO ReLU)
    │
    └─(+) shortcut            : [B,  N,  H,  W]  ← residual on NARROW ends
                                                     (only when N==N', stride==1)
  Output: [B, N', H', W']
```

### 4.3 Residual on the Narrow Ends: Memory Bandwidth Efficiency

The shortcut in a standard ResNet bottleneck connects tensors of shape $[B, C, H, W]$ — the *wide* representation. In MobileNetV2 the shortcut connects $[B, N, H, W]$ — the *narrow* bottleneck. This is not incidental.

On mobile hardware, **memory bandwidth is the dominant cost**, not arithmetic. The inverted residual structure means that the tensors which must be held alive across the full block — the input (for the residual addition) and the output — are both in the compressed bottleneck representation. The wide expanded tensor $[B, tN, H, W]$ exists only transiently and can be freed as soon as the depthwise stage completes.

Compare:

| | Standard ResNet Block | MobileNetV2 Inverted Residual |
|---|---|---|
| Shortcut tensor size | $B \times C \times H \times W$ (wide) | $B \times N \times H \times W$ (narrow) |
| Peak activation memory | Wide tensors must live across block | Only bottleneck tensors persist |
| 3×3 conv channel depth | Narrow ($C/4$) | **Wide ($tN$)** |
| 3×3 conv type | Full (quadratic cost) | **Depthwise (linear cost)** |
| Final activation | ReLU | **None (linear bottleneck)** |

### 4.4 The Expansion Ratio `t` and Its Effect on the Block

The expansion ratio $t$ is the single hyperparameter governing channel width in the expanded space. In the original MobileNetV2 paper, $t = 6$ is the universal choice (except for the first block where $t = 1$, i.e., no expansion).

MACs for one block with input $N$ channels, output $N'$ channels, expansion $t$, output spatial $H' \times W'$:
$$\text{MACs}_{\text{IRB}} = H'W' \cdot tN \cdot \left[ N + K^2 + N' \right]$$
where the three terms correspond to Expand, Depthwise, and Project respectively.

When $t = 1$, the expansion stage is a no-op: `expanded_channels == in_channels`. The block degenerates to DW → Project. The MobileNetV2 paper explicitly handles this case by **omitting the expansion Conv-BN-ReLU6 entirely** — not by instantiating it and passing through a 1×1 identity. We replicate this in our implementation.

In [ ]:
class MobileNetV2Bottleneck(nn.Module):
    """
    MobileNetV2 Inverted Residual Bottleneck Block.

    Implements the Expand → Depthwise → Project (linear) pipeline described in
    Sandler et al. 2018 (https://arxiv.org/abs/1801.04381), Section 3.

    Pipeline
    --------
    Given input channels N, expansion ratio t, output channels N':

      expanded = N * t

    ── Stage 1: Expansion (omitted when t == 1) ─────────────────────────────
    Input     : [B, N,        H,  W ]
    PW Conv2d : [B, expanded, H,  W ]   kernel=1×1; lifts to high-dim space
    BN        : [B, expanded, H,  W ]
    ReLU6     : [B, expanded, H,  W ]   safe: ReLU in high-dim; no collapse

    ── Stage 2: Depthwise Spatial Convolution ───────────────────────────────
    DW Conv2d : [B, expanded, H', W']   kernel=3×3; groups=expanded; stride=s
    BN        : [B, expanded, H', W']
    ReLU6     : [B, expanded, H', W']   safe: ReLU in high-dim; no collapse

    ── Stage 3: Linear Projection (NO activation) ───────────────────────────
    PW Conv2d : [B, N',       H', W']   kernel=1×1; compresses to bottleneck
    BN        : [B, N',       H', W']   affine-only; preserves manifold
    (no act)  :                         ReLU here would collapse low-dim repr.

    ── Residual Addition (conditional) ─────────────────────────────────────
    (+) x     : [B, N',       H', W']   element-wise add of input shortcut
                                         ONLY when stride==1 AND N==N'
    Output    : [B, N',       H', W']

    Parameters
    ----------
    in_channels   : N — bottleneck input channel count.
    out_channels  : N' — bottleneck output channel count.
    stride        : Stride for the depthwise conv. 1 keeps spatial dims;
                    2 halves H and W (used for feature map downsampling).
    expand_ratio  : t — channel multiplier for the expansion stage.
                    When t==1, the expansion Conv-BN-ReLU6 is omitted entirely.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        expand_ratio: int = 6,
    ) -> None:
        super().__init__()

        assert stride in (1, 2), f"stride must be 1 or 2, got {stride}"
        assert expand_ratio >= 1, f"expand_ratio must be ≥ 1, got {expand_ratio}"

        self.stride       = stride
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.expand_ratio = expand_ratio

        expanded = in_channels * expand_ratio  # tN — width of the expanded space

        # ── Stage 1: Expansion ────────────────────────────────────────────────
        # Omitted when t==1: no gain from a 1×1 conv that maps N → N.
        # Using nn.Identity() here would be wrong — it would still consume memory
        # for an extra BN and ReLU6 on an unchanged tensor. We skip it entirely.
        if expand_ratio != 1:
            self.expansion = nn.Sequential(
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=expanded,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                    bias=False,           # BN absorbs the additive bias
                ),
                nn.BatchNorm2d(expanded, eps=1e-5, momentum=0.1, affine=True,
                               track_running_stats=True),
                nn.ReLU6(inplace=True),   # Non-linearity safe in high-dim space
            )
        else:
            self.expansion = None         # Explicit None; checked in forward()

        # ── Stage 2: Depthwise Spatial Convolution ────────────────────────────
        # Operates in the expanded (wide) channel space.
        # groups=expanded enforces per-channel independence (no cross-ch mixing).
        # padding=1 preserves spatial dims when stride=1; halves when stride=2.
        self.depthwise = nn.Sequential(
            nn.Conv2d(
                in_channels=expanded,
                out_channels=expanded,
                kernel_size=3,
                stride=stride,
                padding=1,                # 'same' padding for 3×3; stride controls H'/W'
                groups=expanded,          # depthwise: one 3×3 kernel per channel
                bias=False,
            ),
            nn.BatchNorm2d(expanded, eps=1e-5, momentum=0.1, affine=True,
                           track_running_stats=True),
            nn.ReLU6(inplace=True),       # Still in high-dim space; ReLU is safe
        )

        # ── Stage 3: Linear Projection ────────────────────────────────────────
        # Projects from expanded → out_channels. This is the linear bottleneck.
        # Critically: NO activation after BN. A ReLU6 here would collapse the
        # low-dimensional manifold (see Part 3 for the formal argument).
        self.projection = nn.Sequential(
            nn.Conv2d(
                in_channels=expanded,
                out_channels=out_channels,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels, eps=1e-5, momentum=0.1, affine=True,
                           track_running_stats=True),
            # ← No nn.ReLU6 here. This is the defining feature of a linear bottleneck.
        )

        # ── Residual Connection Gate ──────────────────────────────────────────
        # The shortcut is element-wise addition: requires identical tensor shapes.
        # Both conditions must hold simultaneously:
        #   (a) stride == 1: spatial dimensions H', W' must equal H, W
        #   (b) in_channels == out_channels: channel count must match
        # If either fails, we skip the shortcut entirely — no projection conv is
        # added (unlike ResNet, which uses a 1×1 'downsample' conv to match dims).
        # MobileNetV2 explicitly avoids shortcut projections to save parameters.
        self.use_residual: bool = (stride == 1) and (in_channels == out_channels)

        self._init_weights()

    # ─────────────────────────────────────────────────────────────────────────

    def _init_weights(self) -> None:
        """
        Kaiming uniform for all Conv2d layers; γ=1, β=0 for all BatchNorm layers.

        The projection BN's γ is intentionally initialised to 1 (not 0) to keep
        the activation scale consistent at init. In the residual branch the block
        contributes F(x) ≈ BN(PW(DW(expand(x)))) which is O(1) at init — the
        residual addition y = x + F(x) is dominated by x, which provides stable
        early-training gradients without requiring γ=0 tricks.
        """
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)    # γ = 1 for all BN layers including projection
                nn.init.zeros_(m.bias)     # β = 0

    # ─────────────────────────────────────────────────────────────────────────

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        x   : [B, N,  H,  W]  — input feature map at bottleneck width N

        Returns
        -------
        out : [B, N', H', W'] — inverted residual block output
                                 H' = H if stride=1, else H//2
        """
        identity = x  # Stash for the residual addition before any transformation

        # ── Stage 1: Expand ──────────────────────────────────────────────────
        if self.expansion is not None:
            x = self.expansion(x)   # [B, N,  H, W] → [B, tN, H, W]
        # When expansion is None (t==1): x stays [B, N, H, W] ≡ [B, tN, H, W]

        # ── Stage 2: Depthwise ───────────────────────────────────────────────
        x = self.depthwise(x)       # [B, tN, H, W] → [B, tN, H', W']

        # ── Stage 3: Linear Projection ───────────────────────────────────────
        x = self.projection(x)      # [B, tN, H', W'] → [B, N', H', W']

        # ── Residual Addition ────────────────────────────────────────────────
        if self.use_residual:
            # Shapes are guaranteed identical by the gate conditions checked at init.
            # No projection conv, no channel padding — pure element-wise addition.
            x = x + identity        # [B, N', H', W'] + [B, N, H, W]  (N'==N, H'==H)

        return x

    # ─────────────────────────────────────────────────────────────────────────

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"in={self.in_channels}, out={self.out_channels}, "
            f"stride={self.stride}, t={self.expand_ratio}, "
            f"residual={self.use_residual})"
        )

    @staticmethod
    def count_params(in_channels: int, out_channels: int,
                     expand_ratio: int = 6) -> dict:
        """Analytical conv-weight parameter count (no BN affine)."""
        expanded = in_channels * expand_ratio
        expand_p = (in_channels * expanded) if expand_ratio != 1 else 0
        dw_p     = expanded * 3 * 3            # one 3×3 filter per expanded channel
        proj_p   = expanded * out_channels
        return {
            "expand": expand_p, "depthwise": dw_p,
            "project": proj_p,  "total": expand_p + dw_p + proj_p,
        }

    @staticmethod
    def count_macs(in_channels: int, out_channels: int,
                   h_out: int, w_out: int, expand_ratio: int = 6) -> dict:
        """Analytical MAC count per forward pass, split by stage."""
        expanded  = in_channels * expand_ratio
        expand_m  = (h_out * w_out * in_channels * expanded) if expand_ratio != 1 else 0
        dw_m      = h_out * w_out * expanded * 9             # K²=9
        proj_m    = h_out * w_out * expanded * out_channels
        return {
            "expand": expand_m, "depthwise": dw_m,
            "project": proj_m,  "total": expand_m + dw_m + proj_m,
        }

### 4.5 Anatomising the Two Key Configurations

The block behaves differently in two distinct operational modes, and understanding both is essential before stacking blocks into a full backbone.

**Mode A — Identity residual (`stride=1`, `in_channels == out_channels`)**

This is the standard repeating configuration used in the middle stages of MobileNetV2. The residual connection is active, and the block learns a perturbation on top of the identity. Input and output have the same shape: `[B, N, H, W]`. The expanded tensor `[B, tN, H, W]` exists only transiently inside the block.

**Mode B — Downsampling (`stride=2`, or `in_channels != out_channels`)**

This is the transition configuration used when the feature map resolution is halved or the channel count changes. The residual connection is **disabled** — there is no shortcut, no identity addition. The block acts as a pure feed-forward transform: `[B, N, H, W] → [B, N', H/2, W/2]`. This is also the configuration for the first block of each stage in MobileNetV2's specification table.

The absence of a projection shortcut (unlike ResNet's `downsample` conv) is a deliberate choice: it saves parameters and avoids adding a second memory-bandwidth-intensive path through the block.

In [ ]:
# ── Configuration matrix: test all meaningful combinations ────────────────────
configs = [
    # (in_ch, out_ch, stride, t,  description)
    (16,  16, 1, 6, "Mid-stage: identity residual active"),
    (16,  24, 1, 6, "Channel expansion: no residual (N≠N')"),
    (24,  24, 2, 6, "Downsampling: no residual (stride=2)"),
    (32,  32, 1, 1, "t=1 block: expansion stage omitted"),
    (96, 160, 2, 6, "Late-stage transition block"),
]

B_test, H_test, W_test = 2, 28, 28

print(f"{'Config':45s}  {'residual':>8s}  {'input':>17s}  {'output':>17s}")
print("-" * 100)

for in_ch, out_ch, s, t, desc in configs:
    block = MobileNetV2Bottleneck(
        in_channels=in_ch, out_channels=out_ch,
        stride=s, expand_ratio=t,
    ).to(DEVICE)

    H_in = H_test
    W_in = W_test
    x    = torch.randn(B_test, in_ch, H_in, W_in, device=DEVICE)
    out  = block(x)

    H_out_expected = H_in // s
    W_out_expected = W_in // s
    assert out.shape == (B_test, out_ch, H_out_expected, W_out_expected), \
        f"Shape mismatch for {desc}: got {out.shape}"

    print(
        f"  {desc:43s}  {str(block.use_residual):>8s}  "
        f"{str(list(x.shape)):>17s}  {str(list(out.shape)):>17s}"
    )

In [ ]:
# ── Validate the t=1 path: expansion Sequential must be None ─────────────────
block_t1 = MobileNetV2Bottleneck(in_channels=32, out_channels=32,
                                  stride=1, expand_ratio=1).to(DEVICE)
assert block_t1.expansion is None, "t=1 block must omit the expansion stage"
print(f"t=1 block: expansion stage = {block_t1.expansion}  ✓  (correctly omitted)")

# ── Validate the linear projection: ensure no ReLU in projection stage ────────
block_std = MobileNetV2Bottleneck(in_channels=16, out_channels=16,
                                   stride=1, expand_ratio=6).to(DEVICE)
proj_activations = [m for m in block_std.projection.modules()
                    if isinstance(m, (nn.ReLU, nn.ReLU6))]
assert len(proj_activations) == 0, "Projection stage must be activation-free!"
print(f"Projection stage activations: {proj_activations}  ✓  (linear bottleneck confirmed)")

# ── Print the projection Sequential to show Conv → BN (no activation) ─────────
print(f"\nProjection layers:")
for i, layer in enumerate(block_std.projection):
    print(f"  [{i}] {layer}")

In [ ]:
# ── Gradient flow through the residual path ───────────────────────────────────
# For a stride=1, in==out block, both the residual path and the main branch
# must receive gradients. We verify this by inspecting the input gradient —
# if the residual path is wired correctly, grad w.r.t. x has two additive
# components: one from the identity shortcut (always 1.0 per element) and
# one from the block's learned transform.

block_res = MobileNetV2Bottleneck(in_channels=16, out_channels=16,
                                   stride=1, expand_ratio=6).to(DEVICE)
block_res.train()

x_res = torch.randn(2, 16, 28, 28, device=DEVICE, requires_grad=True)
out_res = block_res(x_res)
out_res.sum().backward()

assert x_res.grad is not None
assert not x_res.grad.isnan().any()

# All block parameters must have gradients
missing = [n for n, p in block_res.named_parameters()
           if p.requires_grad and p.grad is None]
assert len(missing) == 0, f"Missing gradients: {missing}"

print("Gradient flow check — residual block (stride=1, in==out)")
print(f"  Input gradient norm   : {x_res.grad.norm().item():.4f}")
print(f"  All param grads exist : ✓")
print()

# Repeat for the no-residual case
block_ds = MobileNetV2Bottleneck(in_channels=16, out_channels=24,
                                  stride=2, expand_ratio=6).to(DEVICE)
block_ds.train()
x_ds = torch.randn(2, 16, 28, 28, device=DEVICE, requires_grad=True)
block_ds(x_ds).sum().backward()

assert x_ds.grad is not None and not x_ds.grad.isnan().any()
print("Gradient flow check — downsampling block (stride=2, in≠out)")
print(f"  Input gradient norm   : {x_ds.grad.norm().item():.4f}")
print(f"  All param grads exist : ✓")

In [ ]:
# ── Three-way efficiency comparison: std conv vs DSConv vs IRB ────────────────
IN_CH, OUT_CH, H_E, W_E, T = 32, 32, 28, 28, 6

std_p = StandardConv.count_params(IN_CH, OUT_CH, 3)
dsc_p = DepthwiseSeparableConv.count_params(IN_CH, OUT_CH, 3)
irb_p = MobileNetV2Bottleneck.count_params(IN_CH, OUT_CH, T)

std_m = StandardConv.count_macs(IN_CH, OUT_CH, H_E, W_E, 3)
dsc_m = DepthwiseSeparableConv.count_macs(IN_CH, OUT_CH, H_E, W_E, 3)
irb_m = MobileNetV2Bottleneck.count_macs(IN_CH, OUT_CH, H_E, W_E, T)

print(f"Comparing C_in={IN_CH}, C_out={OUT_CH}, spatial={H_E}×{W_E}, t={T}")
print()
print(f"{'Metric':35s} {'StandardConv':>13s}  {'DSConv':>13s}  {'IRB (t=6)':>13s}")
print("-" * 80)
print(f"{'Conv parameters':35s} {std_p:>13,}  {dsc_p['total']:>13,}  {irb_p['total']:>13,}")
print(f"{'  └─ vs StandardConv':35s} {'1.000×':>13s}  {dsc_p['total']/std_p:>12.3f}×  {irb_p['total']/std_p:>12.3f}×")
print()
print(f"{'MACs (M)':35s} {std_m/1e6:>12.3f}M  {dsc_m['total']/1e6:>12.3f}M  {irb_m['total']/1e6:>12.3f}M")
print(f"{'  └─ vs StandardConv':35s} {'1.000×':>13s}  {dsc_m['total']/std_m:>12.3f}×  {irb_m['total']/std_m:>12.3f}×")
print()
print(f"IRB stage breakdown (MACs):")
for stage, macs in irb_m.items():
    if stage != "total":
        pct = 100 * macs / irb_m["total"]
        print(f"  {stage:12s}: {macs/1e6:6.3f}M  ({pct:5.1f}% of block total)")

---

## Summary: Stages 1–4

We have now built every primitive that MobileNetV2 depends on, from the ground up:

| Module | Role | Residual | Activation at output |
|---|---|---|---|
| `StandardConv` | Baseline 3×3 conv-BN-ReLU6 | — | ReLU6 |
| `DepthwiseSeparableConv` | Factorized spatial + channel mixing | — | ReLU6 |
| `MobileNetV2Bottleneck` | Expand → DW → Project (linear) | Conditional | **None** |

The three non-obvious design choices in `MobileNetV2Bottleneck` — and the exact reasoning behind each — are:

1. **Expand first, depthwise second.** The DW conv operates in the *wide* tN space. This is cheap because DW cost scales linearly with channel count, and it gives the spatial filter a richer representation to work with.

2. **No activation on the projection.** The projection compresses tN → N', creating a low-dimensional bottleneck. ReLU applied here would collapse the manifold structure (Part 3). The linear projection preserves it.

3. **Residual only on matching shapes.** The shortcut connects narrow bottleneck tensors — the smallest tensors in the block — minimising persistent memory footprint. No projection conv is added when shapes differ.

### What's Next

**Stage 5** assembles these blocks into the full MobileNetV2 backbone following the specification table from the paper. We will implement the stem conv, all seven inverted residual stages with their exact `(t, c, n, s)` configurations, and the head classifier. The full model will be verified against the published parameter count of 3.4M and the 300M MAC budget for 224×224 input.